In [1]:
import requests
import io
import zipfile
import tempfile
import os
import pandas as pd
from dbfread import DBF

In [ ]:
def cargar_url_en_df(url: str, nombre: str) -> pd.DataFrame:
    """
    Descarga un archivo desde una URL directamente en memoria
    y lo retorna como DataFrame.
    Soporta: zip + (dbf, xlsx, xls, csv)
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, allow_redirects=True, timeout=60)
    response.raise_for_status()
    print(f"[{nombre}] Descargado: {len(response.content)/1024/1024:.2f} MB | Content-Type: {response.headers.get('Content-Type','?')}")

    content = response.content
    buf = io.BytesIO(content)

    # --- ZIP (puede contener DBF, xlsx u otros) ---
    if zipfile.is_zipfile(io.BytesIO(content)):
        with zipfile.ZipFile(buf) as z:
            nombres = z.namelist()
            print(f"[{nombre}] Archivos en ZIP: {nombres}")

            # Buscar DBF dentro del ZIP
            dbf_files = [n for n in nombres if n.lower().endswith(".dbf")]
            xlsx_files = [n for n in nombres if n.lower().endswith(".xlsx")]
            xls_files  = [n for n in nombres if n.lower().endswith(".xls")]
            csv_files  = [n for n in nombres if n.lower().endswith(".csv")]

            if dbf_files:
                dbf_bytes = z.read(dbf_files[0])
                with tempfile.NamedTemporaryFile(suffix=".dbf", delete=False) as tmp:
                    tmp.write(dbf_bytes)
                    tmp_path = tmp.name
                try:
                    tabla = DBF(tmp_path, encoding="latin-1", ignore_missing_memofile=True)
                    df = pd.DataFrame(iter(tabla))
                finally:
                    os.unlink(tmp_path)

            elif xlsx_files:
                df = pd.read_excel(io.BytesIO(z.read(xlsx_files[0])), engine="openpyxl")
            elif xls_files:
                df = pd.read_excel(io.BytesIO(z.read(xls_files[0])), engine="xlrd")
            elif csv_files:
                raw = z.read(csv_files[0])
                try:
                    df = pd.read_csv(io.BytesIO(raw), encoding="utf-8")
                except UnicodeDecodeError:
                    df = pd.read_csv(io.BytesIO(raw), encoding="latin-1")
            else:
                raise ValueError(f"[{nombre}] ZIP no contiene DBF, xlsx, xls ni csv. Archivos: {nombres}")

    # --- Excel directo ---
    elif content[0:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":
        df = pd.read_excel(buf, engine="xlrd")

    # --- CSV directo ---
    else:
        try:
            df = pd.read_csv(buf, encoding="utf-8", sep=None, engine="python")
        except UnicodeDecodeError:
            df = pd.read_csv(io.BytesIO(content), encoding="latin-1", sep=None, engine="python")

    print(f"[{nombre}] shape: {df.shape}")
    return df

In [10]:
df_edif = cargar_url_en_df("https://escale.minedu.gob.pe/documents/10156/e36efe68-64b2-4302-b715-0c4b6037d67b", 'Data_edific')
df_aula = cargar_url_en_df("https://escale.minedu.gob.pe/documents/10156/f5035329-93b9-4fd9-bb4c-1e83dcfa91da", 'Data_aula')
df_padron = cargar_url_en_df("https://escale.minedu.gob.pe/documents/10156/aec96875-bdd2-4d3b-a43a-fb7b89d7738c", 'Data_padron')

[Data_edific] Descargado: 3.89 MB | Content-Type: application/zip
[Data_edific] Archivos en ZIP: ['Loc_P53_Edific.dbf']
[Data_edific] shape: (202417, 84)
[Data_aula] Descargado: 6.87 MB | Content-Type: application/zip
[Data_aula] Archivos en ZIP: ['Loc_P61_Aulas.dbf']
[Data_aula] shape: (268839, 96)
[Data_padron] Descargado: 10.72 MB | Content-Type: application/zip
[Data_padron] Archivos en ZIP: ['Padron.dbf']
[Data_padron] shape: (112724, 59)


In [11]:
df_edif.head(3)

,CODLOCAL,NROCED,CUADRO,NUMERO,ID_EDIF,ID_TERR,P53_3,P53_4,P53_5_1,P53_5_2,...,P53_25_3,P53_26_1,P53_26_2,P53_26_3,P53_27_1,P53_27_2,P53_27_3,FEC_ENVIO,GESTION,ANEXO
0,000019,11,C053,1,E01,TE01,1,2,,,...,2024,0,0,,0,0,,2025-09-15 11:02:09,1,0
1,000024,11,C053,1,E01,TE01,1,2,,,...,2025,0,0,,0,0,,2025-09-26 15:04:52,1,0
2,000024,11,C053,2,E02,TE01,1,2,,,...,,0,0,,0,0,,2025-09-26 15:04:52,1,0


In [12]:
df_aula.head(3)

,CODLOCAL,NROCED,CUADRO,NUMERO,ID_EDIF,P61_2,P61_3,P61_4,P61_5_11,P61_5_12,...,P61_22_81,P61_22_82,P61_22_83,P61_22_91,P61_22_92,P61_22_93,FEC_ENVIO,GESTION,ANEXO,AREA_CENSO
0,000019,11,C061,1,E01,1,1,02,A2,3,...,0,0,0,0,1,0,2025-09-15 11:02:09,1,0,1
1,000019,11,C061,2,E01,1,1,02,A2,5,...,0,0,0,0,1,0,2025-09-15 11:02:09,1,0,1
2,000019,11,C061,3,E01,1,1,02,A2,5,...,0,0,0,0,1,0,2025-09-15 11:02:09,1,0,1


In [13]:
df_padron.head(3)

,CODINST,COD_MOD,ANEXO,CODLOCAL,CEN_EDU,NIV_MOD,D_NIV_MOD,D_FORMA,TIPSSEXO,D_TIPSSEXO,...,INTE_VRAEM,TALUM,TALUM_HOM,TALUM_MUJ,TDOC,TSEC,COD_CAR,D_COD_CAR,IMPUTADO,DES_IMPUTA
0,24953981,0415547,0,016100,123,A2,Inicial - Jardín,Escolarizada,3,Mixto,...,No VRAEM,455,218,237,18,16,a,No aplica,1_INFORMANTE,INFOR.MAT_INFOR.DOC
1,25273230,0415638,0,015172,122,A2,Inicial - Jardín,Escolarizada,3,Mixto,...,No VRAEM,433,214,219,20,18,a,No aplica,1_INFORMANTE,INFOR.MAT_INFOR.DOC
2,20414731,0415646,0,015186,233,A2,Inicial - Jardín,Escolarizada,3,Mixto,...,No VRAEM,589,297,292,26,24,a,No aplica,1_INFORMANTE,INFOR.MAT_INFOR.DOC


In [18]:
cols_edif = ['CODLOCAL', 'ID_TERR', 'P53_3', 'P53_4', 'P53_9', 'P53_10', 'P53_14_3', 'P53_18_1', 'P53_13_1',
             'P53_13_2', 'P53_13_3']
cols_aula = ['CODLOCAL', 'P61_2', 'P61_3', 'P61_4']
cols_padron = ['COD_MOD', 'CODLOCAL', 'D_FORMA','DAREACENSO','TALUM', 'TDOC']

In [20]:
df_padron[cols_padron].head(3)

,COD_MOD,CODLOCAL,D_FORMA,DAREACENSO,TALUM,TDOC
0,0415547,016100,Escolarizada,Urbana,455,18
1,0415638,015172,Escolarizada,Urbana,433,20
2,0415646,015186,Escolarizada,Urbana,589,26


In [21]:
df_edif[cols_edif].head(3)

,CODLOCAL,ID_TERR,P53_3,P53_4,P53_9,P53_10,P53_14_3,P53_18_1,P53_13_1,P53_13_2,P53_13_3
0,000019,TE01,1,2,2,02,2,02,1,1,1
1,000024,TE01,1,2,2,01,1,01,1,1,1
2,000024,TE01,1,2,2,01,1,01,1,1,1


In [22]:
df_aula[cols_aula].head(3)

,CODLOCAL,P61_2,P61_3,P61_4
0,000019,1,1,02
1,000019,1,1,02
2,000019,1,1,02
